# VOICE-CUE sLLM 파인튜닝 — Colab T4

기획서 4-다② "폐쇄망 구동이 가능한 오픈소스 한국어 특화 sLLM을 LoRA로 학습" 단계를 실행합니다.

**시작 전에 반드시**: 런타임 &gt; 런타임 유형 변경 &gt; 하드웨어 가속기 **T4 GPU** 선택

아래 셀을 위에서부터 순서대로 실행하면 됩니다. 총 소요 시간은 3번 셀의 설정에 따라 달라집니다.

## 1. GPU 확인

T4가 아니면 아래 설정값을 조정해야 하므로 먼저 확인합니다.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

import subprocess, sys
out = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout
if not out.strip():
    sys.exit("GPU가 없습니다. 런타임 > 런타임 유형 변경에서 T4 GPU를 선택하세요.")
print(out)

## 2. 저장소 클론 + 의존성 설치

설치에 3~5분 걸립니다. 설치 후 런타임 재시작 안내가 뜨면 무시하고 다음 셀로 진행하세요.

In [ ]:
%cd /content
!rm -rf voiceq-air3
!git clone --branch feature/finetune --depth 1 https://github.com/MeinBau/voiceq-air3.git
%cd /content/voiceq-air3

!pip install -q -r finetune/requirements-train.txt

import importlib
for mod in ("torch", "transformers", "trl", "peft", "bitsandbytes", "datasets", "accelerate"):
    try:
        print(f"{mod:14s} {importlib.import_module(mod).__version__}")
    except Exception as e:
        print(f"{mod:14s} 로드 실패: {e}")

## 3. 학습 설정

T4(16GB)에서의 실측 기반 예상 소요 시간입니다. **무료 Colab은 세션이 끊길 수 있으므로 처음에는 1.5B / 2에폭을 권장**합니다.

| 모델 | 4bit 크기 | 1 에폭 | 2 에폭 | 3 에폭 |
|---|---|---|---|---|
| `Qwen/Qwen2.5-1.5B-Instruct` | ~1.1GB | 약 1.2h | **약 2.4h** | 약 3.6h |
| `Qwen/Qwen2.5-3B-Instruct` | ~2.0GB | 약 2.4h | 약 4.8h | 약 7.2h |

기획서 3-다의 "4bit 양자화 2~4GB" 목표에 정확히 맞는 것은 3B입니다. 1.5B로 먼저 파이프라인이 도는 것을
확인한 뒤, 최종 수치는 3B로 다시 돌리는 순서를 권장합니다(3B는 L4/A100 또는 Colab Pro에서).

학습은 에폭마다 체크포인트를 남기고, 세션이 끊겨도 같은 셀을 다시 실행하면 이어서 학습합니다.

In [ ]:
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"   # 3B로 바꾸려면: "Qwen/Qwen2.5-3B-Instruct"
EPOCHS = 2
BATCH_SIZE = 2
GRAD_ACCUM = 8      # 실효 배치 = BATCH_SIZE * GRAD_ACCUM = 16
MAX_SEQ_LEN = 2048  # 데이터 p99가 1699라 잘림 없음

# 세션이 끊겨도 체크포인트가 남도록 Google Drive에 저장하려면 True.
# (드라이브 마운트 권한 창이 뜹니다. False면 /content에 저장되어 세션 종료 시 사라집니다.)
USE_DRIVE = True

import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/voicecue-finetune"
else:
    OUT = "/content/voiceq-air3/finetune/out"
os.makedirs(OUT, exist_ok=True)
print("체크포인트 경로:", OUT)

## 4. 학습 데이터 생성

데이터는 저장소에 커밋되어 있지 않고 시드 고정 생성물입니다(`--seed 20260829` 기본값).
여기서 만든 것과 로컬에서 만든 것이 완전히 같습니다.

In [ ]:
%cd /content/voiceq-air3
!python finetune/gen_dataset.py

## 5. 학습 전 점검

학습을 몇 시간 돌린 뒤에 데이터가 깨져 있던 걸 발견하면 그 시간을 통째로 버립니다.
먼저 스키마·토큰 길이와 평가 지표 계산식이 정상인지 확인합니다(gold는 100%가 나와야 정상).

In [ ]:
!python finetune/train_lora.py --dry-run --model $MODEL --max-seq-len $MAX_SEQ_LEN
!python tests/test_tiling.py
!python finetune/evaluate.py --backend gold --split test | tail -3

## 6. 베이스라인 측정 (튜닝 전)

파인튜닝의 효과를 말하려면 비교 대상이 있어야 합니다. **튜닝 전 모델에는 few-shot을 켜서** 측정합니다 —
few-shot 없이 형식을 지키게 만드는 것 자체가 파인튜닝의 성과이므로, 양쪽에 똑같이 켜면 그 효과가 측정되지 않습니다.

전체 test 156턴 × 2경로는 시간이 걸리므로 `--limit 40`으로 표본만 봅니다.

In [ ]:
!python finetune/evaluate.py --backend hf --model $MODEL \
    --split test --limit 40 --few-shot \
    --out $OUT/baseline.json

## 7. QLoRA 학습

여기가 오래 걸리는 부분입니다. 10 스텝마다 loss가 찍히고, 에폭마다 체크포인트와 평가 loss가 남습니다.

**세션이 끊기면**: 1·2·3번 셀을 다시 실행한 뒤 이 셀을 다시 실행하세요. 체크포인트를 찾아 이어서 학습합니다
(`USE_DRIVE = True`로 뒀을 때만 세션 종료 후에도 체크포인트가 남습니다).

In [ ]:
!python finetune/train_lora.py \
    --model $MODEL \
    --out $OUT \
    --epochs $EPOCHS \
    --batch-size $BATCH_SIZE \
    --grad-accum $GRAD_ACCUM \
    --max-seq-len $MAX_SEQ_LEN

## 8. 튜닝 모델 평가

튜닝 후에는 **few-shot 없이** 측정합니다. 6번 셀의 베이스라인과 비교하면 파인튜닝 효과가 나옵니다.

In [ ]:
!python finetune/evaluate.py --backend hf --model $MODEL \
    --adapter $OUT/adapter \
    --split test --limit 40 \
    --out $OUT/tuned.json

In [ ]:
import json

base = json.load(open(f"{OUT}/baseline.json", encoding="utf-8"))
tuned = json.load(open(f"{OUT}/tuned.json", encoding="utf-8"))

ROWS = [
    ("지표② 상황유형 정확도",  "지표②_상위배치정확도", "situation_accuracy_pct", "90% 이상"),
    ("지표② COP 셀 일치율",   "지표②_상위배치정확도", "cop_cell_match_pct",    "90% 이상"),
    ("지표④ 키워드 정확도",    "지표④_일지정확도",     "keyword_accuracy_pct",  "90% 이상"),
    ("지표④ 누락률",          "지표④_일지정확도",     "omission_rate_pct",     "5% 미만"),
    ("지표④ 일지 kind 정확도", "지표④_일지정확도",     "log_kind_accuracy_pct", "-"),
    ("지표④ ROUGE-L",        "지표④_일지정확도",     "rouge_l_pct",           "-"),
    ("JSON 유효율 (FAST)",    "부가",                "fast_json_valid_pct",   "-"),
    ("지표① FAST 지연(초)",   "지표①_표출지연",       "fast_p50_sec",          "5초 이내"),
]

print(f"{'지표':24s} {'튜닝 전':>10s} {'튜닝 후':>10s} {'변화':>10s}   목표")
print("-" * 74)
for label, group, key, target in ROWS:
    b, t = base[group][key], tuned[group][key]
    print(f"{label:24s} {b:10.2f} {t:10.2f} {t - b:+10.2f}   {target}")

## 9. 폐쇄망 서빙용 병합 (선택)

어댑터를 베이스에 합쳐 통짜 가중치로 만듭니다. vLLM/Ollama로 부대 내 서버에 올릴 때 쓰며,
앱에서는 사이드바 공급자를 "로컬 서버"로 바꾸기만 하면 코드 수정 없이 연결됩니다.

병합본은 fp16이라 3B 기준 약 6GB입니다. Drive 용량을 확인하고 실행하세요.

In [ ]:
!python finetune/train_lora.py --model $MODEL --out $OUT --merge $OUT/merged
!du -sh $OUT/merged